# **TUTORIAL**: Study objective C (deterministic)

AESA Phase C: absolute sustainability ratio (ASR).\
Compute **deterministic** absolute sustainability ratio (ASR) outputs. ASR = LCA / aCC; the numerator can be IO-LCA or staged external LCA, and
the denominator can include staged external aSoCC methods.

<!-- and run ASR Monte Carlo uncertainty.  -->

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommend to first go through all notebooks available in the `tutorials/core_prerequisites` folder, to have extensive details on what the functions do.

### Functional units and selectors

The example in this tutorial use the **functional unit** `L2.c.b` with the following **selectors**: one producing sector `s_p="Paper"` and one consuming region `r_c="FR"`.

For more details on functional units, please check out:
- the short guide to functional units, region/sector selectors and allocation methods at `tutorials/study_objectives/1_functional_units_and_allocation_methods.md` for a streamlined explanation.
- the dedicated complete documentation in the methodological notes at `methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf` for an in-depth explanation.

In [ ]:
project_name_ = "SO_C_demo_paper_fr"

source_ = "exiobase_3102_ixi"
fu_code_ = "L2.c.b"
study_period_ = range(2010,2021)
s_p_ = ["Paper"]
r_c_ = ["FR"]
lcia_method_ = "pb_lcia"

# Basic features of the deterministic function

### In a nutshell

The function takes necessary **inputs**, mainly for four purposes:
- usual inputs parameters, such as `project_name`, `source`, `years`, `lcia_method`, `fu_code` and the relevant selectors (here `s_p` and `r_c`) for the selected functional unit. `lcia_method` selects the carrying capacity reference used by aCC. Package prerequisites include `pb_lcia`, `gwp100_lcia`, and `ef_3.1`.
- inputs for the with the `lca_args` block, such as `external_lca` and `io_lca`.
- inputs for the deterministic aSoCC function with the `base_asocc_args` block, which mostly includes:
    - All parameters for allocation methods and specific seetings related to enacting metrics can also be reached through nested keys.
- inputs for the deterministic CC function with the `base_cc_args` block, which mostly includes:
    - the `static` block allowing to choose static or dynamic carrying capacities. For dynamic CC, the `dynamic_ar6` block provides parameters and options. Dynamic carrying capacity is available for climate change impact categories
such as `gwp100_lcia` and the EF3.1 `GWP_100` impact category. 


The deterministic **output** folder contains:
- absolute sustainability ratio tables computed as `ASR = LCA / aCC`. Required aCC and IO-LCA prerequisites are created or reused when needed; staged external LCA is used when selected.
- **Figures** are rendered when requested, as controlled with the `figures` and `figure_format` parameters. One additional parameter `figure_options` allows to restrain figures generation to a subset of figures families. The parameter `subfigures` allows to render (or not) figures in the upstream functions needed for aCC, i.e., aSoCC and CC.
- log files

The deterministic output folder contains  Figures are rendered when requested.

Basic features also involve:
- **grouping** of sectors and/or regions: use the `group_sec`, `group_reg`, and `group_version` parameters.
- **overwriting** of existing values: use the `refresh` parameter.
 

<p style="color:#b00020"><strong>IN PROGRESS:</strong> put the allocation paths figure from the PDF here, limited to the L1 and L2 parts. Optionnally, add the PDF allocation method list table for each level and functional unit with references and code syntax.</p>


Allocation method definitions and mathematical expressions are documented in
`data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf`.

`method_plan="default"` applies all <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span> allocation methods available for the
selected functional unit. This is the **recommended** mode to include the widest variety of allocation methods.\
Use `method_plan`, `l1_methods`,
`one_step_methods`, `two_step_methods`, or `l1_l2_pairs` only when a study
needs a smaller method scope.

| Level | Examples | Inputs |
| --- | --- | --- |
| L1 | `EG(Pop)`, `PR(GDPcap)`, `AR(Ecap)`, `PR-HR(Ecap)` | Population, GDP, and optional LCIA stressors. |
| one step L2 | `UT(TD)`, `AR(E)` | Direct L2 vs global allocation. |
| two step L2 | `EG(Pop)_UT(FDa)`, `PR(GDPcap)_UT(GVAa)` | L1 weight multiplied by L2 in L1 weight. |

`lcia_method` is required when LCIA based allocation methods are included.

### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>deterministic_asr(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>project_name</code></td><td>Required project name used to build<br>&#10;<code>&lt;repo&gt;/&lt;project_name&gt;</code>.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>source</code></td><td>MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;<code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;<code>&quot;exiobase_3102_pxp&quot;</code>, or <code>&quot;oecd_v2025&quot;</code>), or <code>&quot;iso3&quot;</code><br>&#10;for ISO3 only mode (L1 EG/PR(GDPcap) only).</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>group_reg</code></td><td>If <code>True</code>, aggregate regions using a grouping file.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source regions.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>group_sec</code></td><td>If <code>True</code>, aggregate sectors using a grouping file.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source sectors.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>group_version</code></td><td>Grouping version tag used to resolve the region/sector<br>&#10;mapping CSVs. Required when <code>group_reg</code> or <code>group_sec</code> is True.<br>&#10;<strong>Defaults to</strong> an empty string for ungrouped processing. Follow<br>&#10;<code>README_grouping.txt</code> in the active<br>&#10;<code>data_raw/mrio/&lt;source&gt;/grouping</code> folder to name grouping<br>&#10;versions and place the matching mapping CSVs.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>years</code></td><td>Studied years. Accepts a single year, list, or range. If<br>&#10;<strong>omitted</strong>, all available MRIO<br>&#10;years for the selected source/group version are used.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>lcia_method</code></td><td>Shared downstream LCIA method name or list of names. When<br>&#10;static carrying capacity (CC) is active, the denominator requires<br>&#10;a matching carrying capacity CSV. The package includes static CC files for<br>&#10;<code>&quot;pb_lcia&quot;</code>, <code>&quot;gwp100_lcia&quot;</code>, and <code>&quot;ef_3.1&quot;</code>; among these,<br>&#10;<code>&quot;ef_3.1&quot;</code> is the <strong>default</strong> carrying capacity method that<br>&#10;currently has no MRIO LCIA characterization matrix. It is still<br>&#10;dynamic AR6 compatible for its <code>&quot;GWP_100&quot;</code> impact category.<br>&#10;Dynamic AR6 CC is supported for <code>&quot;gwp100_lcia&quot;</code> and for<br>&#10;<code>&quot;ef_3.1&quot;</code> impact <code>&quot;GWP_100&quot;</code> only. Custom static CC methods<br>&#10;also require a matching file in<br>&#10;<code>data_raw/carrying_capacities/</code>; follow<br>&#10;<code>README_add_custom_carrying_capacities.txt</code>. When upstream<br>&#10;allocated shares of carrying capacities (aSoCC) LCIA based methods<br>&#10;or IO-LCA generation is in scope, custom EXIOBASE MRIO LCIA<br>&#10;methods must be prepared with<br>&#10;<code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/README_add_custom_lcia_characterization_matrices.txt</code><br>&#10;and processed with <code>process_mrio(...)</code>.<br>&#10;<code>base_asocc_args[&quot;include_lcia_based_allocation_methods&quot;]</code> controls whether<br>&#10;LCIA based allocation methods are included. When it is <code>False</code>,<br>&#10;upstream aSoCC keeps only non LCIA dependent methods, so the requested<br>&#10;<code>lcia_method</code> may be non MRIO when the matching carrying capacity<br>&#10;and selected LCA numerator prerequisites exist.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>fu_code</code></td><td>Required functional unit code (for example <code>&quot;L1.a&quot;</code>,<br>&#10;<code>&quot;L2.c.b&quot;</code>). See<br>&#10;<code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;for all available functional unit codes and the system<br>&#10;boundaries each represents.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>s_p</code></td><td>Producing sector filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid producing sectors. To identify valid sector<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../grouping/.../group_sec_template.csv</code> file. For<br>&#10;EXIOBASE sector definitions, see<br>&#10;<code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>; EXIOBASE<br>&#10;ixi and pxp use different sector lists.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_p</code></td><td>Producing region filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid producing regions. To identify valid region<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../grouping/group_reg_template.csv</code> file.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_c</code></td><td>Consuming region filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid consuming regions. To identify valid region<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../grouping/group_reg_template.csv</code> file.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_f</code></td><td>Final demand region filter(s), single string or list. If this is<br>&#10;a required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the<br>&#10;run expands to all valid final demand regions. To identify valid<br>&#10;region names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../grouping/group_reg_template.csv</code> file.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>aggreg_indices</code></td><td>Whether multiple selected region/sector indices are reported as<br>&#10;separate rows or summed into one row after the selected MRIO scope<br>&#10;is computed.<br>&#10;&bull; <code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;&bull; <code>True</code>: sum selected values into one row.<br>&#10;Not allowed for <code>L2.a.b</code>, <code>L2.b.b</code>, and <code>L2.c.b</code> because<br>&#10;aggregating CBA total demand system boundaries can double count.<br>&#10;For these functional units, define the aggregation from<br>&#10;<code>process_mrio(...)</code> onward with<br>&#10;<code>group_reg</code>/<code>group_sec</code>/<code>group_version</code>.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_asocc_args</code></td><td>Optional aSoCC only envelope reused to resolve the<br>&#10;deterministic upstream aSoCC prerequisite. Write nested arguments<br>&#10;as <code>base_asocc_args={&quot;method_plan&quot;: &quot;default&quot;}</code>. <strong>Omit</strong> the<br>&#10;envelope or any accepted key to use its <strong>default</strong>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>method_plan</code>: <code>method_plan</code> <strong>defaults to</strong> <code>&quot;default&quot;</code> and<br>&#10;  accepts <code>&quot;default&quot;</code>, <code>&quot;one_step&quot;</code>, <code>&quot;two_steps&quot;</code>,<br>&#10;  <code>&quot;pairs&quot;</code>, or <code>&quot;one_step_pairs&quot;</code>. <strong>When omitted</strong>, all pyaesa<br>&#10;  allocation methods available for the selected <code>fu_code</code> are<br>&#10;  applied. See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for the allocation methods available per functional unit,<br>&#10;  including definitions and mathematical expressions.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l1_methods</code>: Optional L1 subset. <strong>Omit</strong> it to keep all L1<br>&#10;  methods allowed by <code>method_plan</code>. In <code>&quot;default&quot;</code>, this<br>&#10;  filters only L1 weights used by two step methods. In<br>&#10;  <code>&quot;two_steps&quot;</code>, it filters the two step cartesian L1 side.</span><br>&#10;<span style="color:#087f5b">&bull; <code>one_step_methods</code>: Optional one step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all one step methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull; <code>two_step_methods</code>: Optional two step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all two step L2 methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l1_l2_pairs</code>: Explicit pair list formatted as<br>&#10;  <code>&quot;L1METHOD::L2METHOD&quot;</code>. <strong>Omit</strong> it unless <code>method_plan</code> is<br>&#10;  <code>&quot;pairs&quot;</code> or <code>&quot;one_step_pairs&quot;</code>.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l1_reg_aggreg</code>: L1 aggregation mode for methods where timing<br>&#10;  matters (<code>PR(GDPcap)</code>, <code>PR-HR(Ecap)</code> and <code>AR(Ecap)</code>).<br>&#10;  <code>&quot;pre&quot;</code> aggregates regions before L1 computation. <code>&quot;post&quot;</code><br>&#10;  (<strong>default</strong>) computes on original regions, then aggregates.</span><br>&#10;<span style="color:#087f5b">&bull; <code>reference_years</code>: Acquired rights (AR) methods reference<br>&#10;  year selector. Accepts a single year, list, or range. If<br>&#10;  <strong>omitted</strong>, AR routes use all historical years in the studied range<br>&#10;  up to the source registry historical cutoff. For EXIOBASE<br>&#10;  3.10.2 and OECD ICIO v2025, the cutoff is 2022; other supported<br>&#10;  MRIO sources use their own registry cutoff.</span><br>&#10;<span style="color:#087f5b">&bull; <code>ssp_scenario</code>: SSP scenario name or list. <strong>Defaults to</strong><br>&#10;  <code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code> and is applied<br>&#10;  when scenario dependent inputs are required.</span><br>&#10;<span style="color:#087f5b">&bull; <code>projection_mode</code>: Projection policy for post historical<br>&#10;  years of L2 utilitarian (UT) methods (MRIO economic enacting<br>&#10;  metrics). <strong>Defaults to</strong> <code>&quot;regression&quot;</code>. <code>&quot;regression&quot;</code><br>&#10;  projects UT inputs for future years. <code>&quot;historical_reuse&quot;</code><br>&#10;  reuses historical UT structures.</span><br>&#10;<span style="color:#087f5b">&bull; <code>reg_window</code>: Historical regression fit window for regression<br>&#10;  mode. Provide it as <code>range(start_year, end_year + 1)</code> or as<br>&#10;  an explicit list of consecutive years in fit window order. When<br>&#10;  <strong>omitted</strong>, the source registry supplies the <strong>default</strong> fit window<br>&#10;  from the modeled year minimum through the source historical<br>&#10;  cutoff. For EXIOBASE 3.10.2 and OECD ICIO v2025, this resolves<br>&#10;  to 1995 to 2022; other supported MRIO sources use their own<br>&#10;  registry window.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l2_reuse_years</code>: Historical L2 reuse year selector used by<br>&#10;  all UT historical reuse routes. In<br>&#10;  <code>projection_mode=&quot;historical_reuse&quot;</code> it applies to all UT<br>&#10;  methods; in <code>projection_mode=&quot;regression&quot;</code> it applies to<br>&#10;  adjusted UT routes (<code>UT(FDa)</code>, <code>UT(GVAa)</code>), which always<br>&#10;  use historical reuse as regression is not supported (would<br>&#10;  require regression on full MRIO). <strong>If omitted</strong>, <strong>defaults to</strong><br>&#10;  <code>reg_window</code> when required.</span><br>&#10;<span style="color:#087f5b">&bull; <code>include_lcia_based_allocation_methods</code>: Whether to include LCIA based<br>&#10;  allocation methods (e.g.: acquired rights - AR, or historical<br>&#10;  responsibility - PR-HR). <strong>Defaults to</strong> <code>True</code>. <code>False</code> keeps only non LCIA dependent<br>&#10;  allocation methods.</span></td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_cc_args</code></td><td>Carrying capacity family envelope. The package <strong>default<br>&#10;is</strong> static active and dynamic AR6 inactive. Provide a<br>&#10;<code>dynamic_ar6</code> block to add dynamic AR6 carrying capacities. Set<br>&#10;<code>static.active=False</code> for dynamic only runs.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>static</code>: Static carrying capacity route. It is active by<br>&#10;  <strong>default</strong> unless <code>active=False</code> is provided.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether the static route is active. <strong>Defaults to</strong><br>&#10;    <code>True</code>.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>exclude_max_cc</code>: Whether to use only <code>min_cc</code>. <strong>Defaults<br>&#10;    to</strong> <code>False</code>. <code>False</code> keeps the paired<br>&#10;    <code>min_cc</code> plus <code>max_cc</code> interpretation when <code>max_cc</code> is<br>&#10;    present. <code>True</code> uses only <code>min_cc</code>.</span><br>&#10;&#160;<br>&#10;<span style="color:#c45f00">&bull; <code>dynamic_ar6</code>: Dynamic AR6 carrying capacity route. It is<br>&#10;  inactive <strong>when omitted</strong>. When the block is provided, it is active<br>&#10;  unless <code>active=False</code> is provided. It uses top level<br>&#10;  <code>years</code> and requires at least two consecutive years.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether the dynamic AR6 route is active.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>harmonization</code>: Whether to harmonize retained AR6 pathways<br>&#10;    to the historical baseline. <strong>Defaults to</strong> <code>True</code>.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>harmonization_method</code>: Harmonization method applied only<br>&#10;    when <code>harmonization=True</code>. <strong>Defaults to</strong> <code>&quot;offset&quot;</code>. The<br>&#10;    only supported value is currently <code>&quot;offset&quot;</code>. Ignored when<br>&#10;    <code>harmonization=False</code>.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>category</code>: AR6 category classification selector for global<br>&#10;    warming trajectories, as a string or list, such as <code>&quot;C2&quot;</code><br>&#10;    or <code>[&quot;C1&quot;, &quot;C2&quot;]</code>. <strong>Defaults to</strong> C1 to C4.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>ssp_scenario</code>: Canonical SSP selector as a string, list,<br>&#10;    or <code>None</code>, such as <code>&quot;SSP2&quot;</code> or <code>[&quot;SSP1&quot;, &quot;SSP2&quot;]</code>.<br>&#10;    <strong>Defaults to</strong> SSP1 to SSP5.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>emission_type</code>: Dynamic AR6 emission type. Accepted values<br>&#10;    are <code>&quot;kyoto_gases&quot;</code> (<strong>default</strong>) and <code>&quot;co2&quot;</code>.<br>&#10;    <code>emission_type=&quot;kyoto_gases&quot;</code> uses the GWP100 Kyoto Gases<br>&#10;    aggregate; <code>emission_type=&quot;co2&quot;</code> uses direct CO2 pathways.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>include_afolu</code>: Whether AFOLU is included inside the<br>&#10;    selected <code>emission_type</code>. <strong>Defaults to</strong> <code>False</code>.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>emissions_mode</code>: Dynamic AR6 emissions mode. Accepted<br>&#10;    values are <code>&quot;net&quot;</code>, <code>&quot;gross&quot;</code>, and <code>&quot;gross_alt&quot;</code>.<br>&#10;    <strong>Defaults to</strong> <code>&quot;gross_alt&quot;</code>. Gross modes write positive<br>&#10;    emissions denominator rows and signed negative sequestration<br>&#10;    companion rows; downstream aCC and ASR consume only the<br>&#10;    denominator gross positive rows. <code>&quot;gross&quot;</code> removes all<br>&#10;    sequestration sources from net emissions. <code>&quot;gross_alt&quot;</code><br>&#10;    removes all sequestration sources except CCS, as it does not<br>&#10;    directly capture CO2 from the atmosphere; IPCC AR6 recommends<br>&#10;    treating CCS separately from net negative sequestration. See<br>&#10;    <code>data_raw/methodological_notes/methodological_note__steady_state__dynamic_cc.pdf</code><br>&#10;    for the methodological explanation.</span><br>&#10;<span style="color:#c45f00">  &bull; <code>subset_version</code>: Optional selector for a subset of AR6<br>&#10;    model-scenario pairs. Follow<br>&#10;    <code>data_processed/ar6/&lt;processed_scope&gt;/README_model_scenario_subset.txt</code><br>&#10;    to create the subset CSV.</span></td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>lca_args</code></td><td>Required LCA numerator route envelope. Exactly one route<br>&#10;block must have <code>active=True</code>. The <strong>default</strong> signature selects<br>&#10;<code>external_lca</code>. Set <code>external_lca.active=False</code> and<br>&#10;<code>io_lca.active=True</code> to use IO-LCA generation.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>external_lca</code>: External LCA route block.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether staged external LCA files under<br>&#10;    <code>A_lca/external_lca/</code> are used.</span><br>&#10;  &bull; <code>version_name</code>: External LCA version selected from staged<br>&#10;    files. Use <code>prepare_external_inputs(...)</code> to import the<br>&#10;    external LCA real input folders, runnable CSV examples, and README guide, then follow the imported guide for version syntax and<br>&#10;    data input format.<br>&#10;&#160;<br>&#10;<span style="color:#c45f00">&bull; <code>io_lca</code>: IO-LCA generation route block.<br>&#10;&#160;<br>&#10;  Nested key:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether IO-LCA generation is used.</span></td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>external_method</code></td><td>Optional external aSoCC method selector. Use<br>&#10;<code>{&quot;l1_methods&quot;: [...]}</code> for L1 functional units. For L2<br>&#10;functional units use <code>{&quot;one_step_methods&quot;: [...]}</code> and/or<br>&#10;<code>{&quot;l1_l2_pairs&quot;: [&quot;&lt;l1_method&gt;::&lt;l2_method&gt;&quot;, ...]}</code>.<br>&#10;<strong>Omit</strong> this argument or pass <code>None</code> when using only native pyaesa<br>&#10;deterministic aSoCC methods. Use <code>prepare_external_inputs(...)</code><br>&#10;to import the external aSoCC runnable CSV examples and README guide, then<br>&#10;follow the imported guide for external method names, selector<br>&#10;syntax, and deterministic or Monte Carlo external aSoCC CSV<br>&#10;preparation.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Persisted output file format: <code>&quot;csv&quot;</code> (<strong>default</strong>),<br>&#10;<code>&quot;pickle&quot;</code>, or <code>&quot;parquet&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_options</code></td><td>Figure product selector mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;per_method&quot;: True, &quot;multi_method&quot;: True}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>per_method</code>: Whether to render method specific figures, with one separate<br>&#10;  figure for each allocation method.<br>&#10;&#160;<br>&#10;&bull; <code>multi_method</code>: Whether to render cross method comparison figures, with<br>&#10;  multiple allocation methods shown in the same figure.<br>&#10;&#160;<br>&#10;Deterministic ASR renders deterministic polar figures for eligible single year, multi impact, static carrying capacity per method scopes. It does not expose <code>polar</code> or <code>polar_years</code> in <code>figure_options</code>; select the polar figure year through the main <code>years</code> argument.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull; <code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>subfigures</code></td><td>Whether prerequisite LCA and dynamic AR6 CC figures are<br>&#10;also rendered during the automatic prerequisite cascade<br>&#10;when <code>figures=True</code>. <strong>Default is</strong> <code>True</code>.</td></tr>
<tr><td colspan="2" style="height:0.75rem;"></td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, clear and recompute the resolved deterministic ASR output scope for this project, source and group version, numerator route, carrying capacity source, and carrying capacity type, plus the matching deterministic aCC prerequisite scope and, when the numerator route is IO-LCA, the IO-LCA prerequisite scope used by that request. Through the deterministic aCC prerequisite, this can include the matching deterministic aSoCC output scope and, when dynamic AR6 CC is used, the matching processed AR6 output scope selected by <code>process_ar6(...)</code> and the matching <code>deterministic_ar6_cc(...)</code> output scope. External LCA staged inputs, processed MRIO inputs, processed population and GDP, and raw downloads are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>


### Deterministic ASR with IO-LCA and static pb_lcia aCC

In [ ]:
from pyaesa import deterministic_asr

deterministic_asr(
    project_name=project_name_,
    source=source_,
    years=study_period_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method="pb_lcia",
    lca_args={
        "external_lca": {"active": False},
        "io_lca": {"active": True},
    },
    refresh=True,
)

### Deterministic ASR with EF 3.1 static aCC and external LCA numerator

Run this after staging the external LCA files with `prepare_external_inputs(...)`. The packaged
external LCA example is ready to run with `version_name="template"` (numeric valuesvare dummy demonstration values).

This example uses the EF 3.1 route: `ef_3.1` has no EXIOBASE LCIA characterization matrix in
pyaesa, so original pyaesa LCIA based allocation methods are disabled with
`base_asocc_args["include_lcia_based_allocation_methods"] = False`.


In [ ]:
from pyaesa import prepare_external_inputs

prepare_external_inputs(project_name=project_name_)

deterministic_asr(
    project_name=project_name_,
    source=source_,
    years=range(2019, 2031),
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method="ef_3.1",
    base_asocc_args={
        # ef_3.1 has no EXIOBASE LCIA characterization matrix in pyaesa.
        # Hence "include_lcia_based_allocation_methods" is False.
        # This excludes LCIA based allocation methods, such as acquired rights (AR),
        # because they require an EXIOBASE LCIA characterization matrix.
        "include_lcia_based_allocation_methods": False,
        "ssp_scenario": ["SSP2"],
    },
    lca_args={
        "external_lca": {
            "active": True,
            "version_name": "template",
        },
    },
    refresh=True,
)



### Deterministic ASR with dynamic AR6 aCC denominator

By default, all figures are rendered by the function. In this case, this can lead to a significant number of figures (several hundreds), as one figure will be generated for each model-scenario. The `figures` parameter is hence set to `False` given the large number of model-scenarios. With a subset of model-scenarios selected, figure rendering is useful in deterministic mode.

In [ ]:
deterministic_asr(
    project_name=f"{project_name_}_prosp",
    source=source_,
    years=study_period_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method="gwp100_lcia",
    base_asocc_args={
        "ssp_scenario": ["SSP2"],
    },
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {
            "active": True,
            "category": ["C1", "C2"],
            "ssp_scenario": ["SSP2"],
        },
    },
    lca_args={
        "external_lca": {"active": False},
        "io_lca": {"active": True},
    },
    figures=False,
    refresh=True,
)

# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at `tutorials/study_objectives/...`

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available includes:
- Custom  external aSoCC

### Custom external aSoCC

Use `external_method` when the aCC denominator should include staged external
aSoCC methods. The external input optional workflow explains the external aSoCC
README guide, filename syntax, and deterministic or Monte Carlo staging folders.

The packaged dummy examples include the one step `CO(S)` route and the two step
`AR(E)::UT(S)` route. In this EF 3.1 example, `CO(S)` is non LCIA specific, while
`AR(E)::UT(S)` is supplied directly as an external EF 3.1 LCIA specific method. pyaesa original
LCIA based allocation methods remain disabled for the same reason as above.


In [ ]:
prepare_external_inputs(project_name=project_name_)

deterministic_asr(
    project_name=project_name_,
    source=source_,
    years=range(2019, 2031),
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method="ef_3.1",
    base_asocc_args={
        # ef_3.1 has no EXIOBASE LCIA characterization matrix in pyaesa.
        # Hence "include_lcia_based_allocation_methods" is False.
        # This excludes native LCIA based allocation methods because they require
        # an EXIOBASE LCIA characterization matrix. The external AR(E)::UT(S)
        # rows below are supplied directly by external input files.
        "include_lcia_based_allocation_methods": False,
        "ssp_scenario": ["SSP2"],
    },
    external_method={
        "one_step_methods": ["CO(S)"],
        "l1_l2_pairs": ["AR(E)::UT(S)"],
    },
    lca_args={
        "external_lca": {
            "active": True,
            "version_name": "template",
        },
    },
    refresh=True,
)

